In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

import sys
sys.path.append('../data')
from utils.functions import load_yaml

In [ ]:
# Load Configuration
config = load_yaml("../config/RRGCL.yaml")
SAVEPATH = config.SAVEPATH

# Cancer Driver DataFrame
mut_df = pd.read_csv(f'{config.DATABASE}/matched_cancer_driver_df.csv')

In [ ]:
models = {
    "All": 'szusj1rl',
    'ExLocalGeo': '3rki24k9',
    'ExLocalGeo+Egy': '3v9dkf3h',
    'All+Egy': 'rtn2y7dv',
    'All+Egy2': 'di9woh2e',
    'ExSpEnv': 'v6iuy1du'
    'ExSpEnv+Egy': 'sryxn215',
    'ExEvol': "dytsoorb",
    'ExEvol+Egy': 'fa2h3zct'
}

# Cancer Driver Visualization

In [ ]:
def plot_tsne_by_label(models_dict, num_cols=3):
    num_models = len(models_dict)
    num_rows = math.ceil(num_models / num_cols)
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 6, num_rows * 6))
    
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.flatten()
    
    for i, (model_name, wandb_id) in enumerate(models_dict.items()):
        ax = axes[i]
        
        vis_df = pd.read_csv(f"{SAVEPATH}/DGI/{wandb_id}/tSNE_result_driver+am.csv")
        vis_df = vis_df[~(vis_df['tsne_1']<30)]
        
        sns.scatterplot(
            x='tsne_1', 
            y='tsne_2',
            hue='label',
            palette='Set2',
            data=vis_df,
            alpha=0.7,
            ax=ax
        )
        ax.set_title(f'{model_name} (Label)')
        ax.set_xlabel('t-SNE 1')
        ax.set_ylabel('t-SNE 2')
        
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

plot_tsne_by_label(models, num_cols=3)

In [ ]:
def plot_tsne_by_max_am(models_dict, num_cols=3):
    num_models = len(models_dict)
    num_rows = math.ceil(num_models / num_cols)
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 6, num_rows * 6))
    
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.flatten()
    
    scatter = None 
    
    for i, (model_name, wandb_id) in enumerate(models_dict.items()):
        ax = axes[i]
        
        vis_df = pd.read_csv(f"{SAVEPATH}/DGI/{wandb_id}/tSNE_result_driver+am.csv")
        vis_df = vis_df[~(vis_df['tsne_1']<30)]
        
        scatter_obj = ax.scatter(
            vis_df['tsne_1'], 
            vis_df['tsne_2'],
            c=vis_df['max_am'],
            cmap='viridis_r',
            alpha=0.7,
            s=30,
            edgecolors='w',
            linewidth=0.2,
            vmin=0.3,
            vmax=0.9
        )
        scatter = scatter_obj
        
        ax.set_title(f'{model_name} (max_am)')
        ax.set_xlabel('t-SNE 1')
        ax.set_ylabel('t-SNE 2')
        
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout(rect=[0, 0, 0.95, 1]) 
    
    if scatter is not None:
        cbar_ax = fig.add_axes([0.96, 0.15, 0.015, 0.7]) 
        cbar = fig.colorbar(scatter, cax=cbar_ax)
        cbar.set_label('Average AlphaMissense Score (max_am)')
        
    plt.show()

plot_tsne_by_max_am(models, num_cols=3)

In [ ]:
vis_org_df = pd.read_csv(f'{SAVEPATH}/DGI/tSNE_result_all-feat_org.csv')
vis_org_df = vis_org_df.merge(mut_df[['node_id', 'label']], on='node_id', how='left')
vis_org_df = vis_org_df[vis_org_df['node_id'].isin(vis_df['node_id'])]

In [ ]:
vis_org_df

In [ ]:
# 6. Visualize by Label and max_am side-by-side in a 1x2 Subplot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# [Left] Scatter plot by Label
sns.scatterplot(
    x='tsne_1', y='tsne_2', hue='label_x', palette='Set2',
    data=vis_org_df, alpha=0.7, ax=axes[0]
)
axes[0].set_title('t-SNE of Original Features by Label')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# [Right] Scatter plot by max_am
scatter = axes[1].scatter(
    vis_org_df['tsne_1'], vis_org_df['tsne_2'],
    c=vis_org_df['max_am'], cmap='viridis_r', alpha=0.7,
    edgecolors='w', linewidth=0.2, s=30
)
axes[1].set_title('t-SNE of Original Features by max_am')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

# Add colorbar
cbar = fig.colorbar(scatter, ax=axes[1])
cbar.set_label('Average AlphaMissense Score (max_am)')
plt.tight_layout()
plt.show()